# 🎯 BioEcoInt — Benchmark Multi-Run YOLO-Seg (Detecção & Segmentação em Tempo Real)

> **Laboratório IFMaker / Projeto BioEcoInt**  
> Notebook Colab focado exclusivamente no **Fine-Tuning & Benchmark Estatístico (3 Runs: Média, Desvio Padrão e Variância)** de modelos leves **YOLO-Seg** sobre o dataset completo [`animals_video_v35_yolo_seg`](../datasets/generated/animals_video_v35_yolo_seg/data.yaml) (`7.277` imagens full-frame + polígonos SAM 2 + *hard negatives* em `fundo_sem_animal`).

**Protocolo Experimental:**
- **Trilha `-seg`:** `yolov8n-seg.pt`, `yolo11n-seg.pt`, `yolo26n-seg.pt` (+ célula opcional `yolo11s-seg.pt`)
- **Repetições:** `3` runs independentes por modelo (`seeds = 42, 142, 242`) a `imgsz = 640`, `epochs = 60`, `patience = 15`, `cos_lr = True`
- **Avaliação Honesta (`split="test"`):** métricas Box & Mask `mAP@50-95`, separação entre **`13` classes disjuntas (`N >= 3`)** vs **`2` compartilhadas (`N <= 2`)**, taxa de FPs em **`fundo_sem_animal`** e exportação comparativa do campeão em **`640px` vs `320px`** (`ONNX` / `TFLite` + FPS).


### 1. Verificação de GPU e Hardware


In [ ]:
import torch
print(f"CUDA disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | VRAM: {props.total_memory / 1e9:.2f} GB")
else:
    raise RuntimeError("❌ Ative uma GPU no menu: Runtime -> Change runtime type -> T4/L4/A100 GPU!")


### 2. Instalação do `uv` e Clonagem / Atualização do Repositório


In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = f"/root/.local/bin:{os.environ.get('PATH', '')}"

from google.colab import userdata

try:
    github_token = userdata.get("GITHUB_TOKEN")
    repo_url = f"https://{github_token}@github.com/Emmerson2018/bio-interaction-ml.git"
except Exception:
    repo_url = "https://github.com/Emmerson2018/bio-interaction-ml.git"

if not os.path.exists("bio-interaction-ml"):
    !git clone {repo_url}
    %cd bio-interaction-ml
else:
    %cd bio-interaction-ml
    !git pull origin main


### 3. Sincronização do Ambiente (`uv sync` + Ultralytics `8.4.199+`)


In [ ]:
!uv sync
!uv run python -c "import torch, ultralytics; print(f'✅ PyTorch {torch.__version__} | Ultralytics {ultralytics.__version__}')" 


### 4. Google Drive, Autenticação DVC (`Colab Secret` ou Upload) & Dataset `animals_video_v35_yolo_seg`

> **Como automatizar o `default.json` via Colab Secret (sem precisar subir toda vez):**  
> 1. Abra a aba **Secrets (`🔑`)** no painel esquerdo do Google Colab;  
> 2. Crie um segredo chamado **`DVC_DEFAULT_JSON`**;  
> 3. Cole todo o conteúdo JSON do seu arquivo `default.json` e habilite **Notebook access**.  
> *(Caso o segredo ainda não esteja configurado, a célula faz fallback automático para `files.upload()`.)*


In [ ]:
from google.colab import drive, files, userdata
import json, os, shutil, subprocess

drive.mount("/content/drive")

creds = None
try:
    raw_secret = userdata.get("DVC_DEFAULT_JSON")
    if raw_secret and str(raw_secret).strip():
        creds = json.loads(str(raw_secret).strip())
        print("🔐 Credenciais DVC carregadas automaticamente via Colab Secret 'DVC_DEFAULT_JSON'.")
except Exception as e:
    print(f"ℹ️ Secret 'DVC_DEFAULT_JSON' indisponível ({e}); usando fallback de upload.")

if creds is None:
    print("📤 Faça o upload do arquivo 'default.json' para autenticar o DVC:")
    uploaded = files.upload()
    json_files = [fn for fn in uploaded if fn.endswith(".json")]
    if not json_files or not os.path.exists(json_files[0]):
        raise RuntimeError("❌ O arquivo 'default.json' é obrigatório para puxar o dataset via DVC.")
    with open(json_files[0], "r", encoding="utf-8") as f:
        creds = json.load(f)
    os.remove(json_files[0])

cid = creds.get("client_id")
cs = creds.get("client_secret")
if not cid or not cs:
    raise ValueError("❌ 'default.json' inválido: 'client_id' ou 'client_secret' ausentes.")

creds_dir = f"/root/.cache/pydrive2fs/{cid}"
os.makedirs(creds_dir, exist_ok=True)
target_file = os.path.join(creds_dir, "default.json")
with open(target_file, "w", encoding="utf-8") as f:
    json.dump(creds, f)
print(f"✅ Credenciais DVC gravadas em {target_file}")

subprocess.run(["uv", "run", "dvc", "remote", "modify", "--local", "gdrive-storage", "gdrive_client_secret", cs], check=True)

LOCAL_DATA_DIR = "datasets/generated"
DS_NAME = "animals_video_v35_yolo_seg"
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
target_dir = os.path.join(LOCAL_DATA_DIR, DS_NAME)
zip_name = f"{DS_NAME}.zip"
local_zip = os.path.join(LOCAL_DATA_DIR, zip_name)
dvc_file = f"{LOCAL_DATA_DIR}/{zip_name}.dvc"

if not (os.path.exists(target_dir) and os.path.exists(os.path.join(target_dir, "data.yaml"))):
    print(f"⬇️ Baixando {zip_name} via DVC...")
    res = subprocess.run(["uv", "run", "dvc", "pull", dvc_file], check=False)
    if res.returncode != 0 or not os.path.exists(local_zip):
        raise RuntimeError(f"❌ Falha ao baixar {dvc_file} via DVC. Rode `dvc add` + `dvc push` na estação local antes.")
    print(f"📂 Descompactando {zip_name}...")
    subprocess.run(["unzip", "-q", "-o", local_zip, "-d", LOCAL_DATA_DIR], check=True)

for sp in ("train", "val", "test"):
    n_imgs = len(os.listdir(os.path.join(target_dir, "images", sp)))
    n_lbls = len(os.listdir(os.path.join(target_dir, "labels", sp)))
    print(f"  [{sp:5s}] imagens={n_imgs} | labels={n_lbls}")
assert os.path.exists(os.path.join(target_dir, "data.yaml"))
print("✅ Dataset YOLO-Seg pronto para benchmark!")


---
### 5. Execução Incremental do Benchmark YOLO-Seg (`3 Runs` por Modelo)

Cada célula usa `--resume` (se o Colab desconectar no meio, basta reexecutar para continuar da run pendente) e `--no-export-champion` até a etapa de consolidação final.


#### 5.1 Modelo 1: `yolov8n-seg.pt` (Baseline CNN C2f — 3 Runs)


In [ ]:
!MPLBACKEND=Agg uv run python experiments/benchmark_yolo.py \
  --data datasets/generated/animals_video_v35_yolo_seg/data.yaml \
  --models yolov8n-seg.pt \
  --runs 3 --base-seed 42 --epochs 60 --imgsz 640 --batch 16 --patience 15 \
  --output-dir experiments/YOLO_Seg_Benchmark \
  --no-export-champion


#### 5.2 Modelo 2: `yolo11n-seg.pt` (C3k2 + Atenção Espacial C2PSA — 3 Runs)


In [ ]:
!MPLBACKEND=Agg uv run python experiments/benchmark_yolo.py \
  --data datasets/generated/animals_video_v35_yolo_seg/data.yaml \
  --models yolo11n-seg.pt \
  --runs 3 --base-seed 42 --epochs 60 --imgsz 640 --batch 16 --patience 15 \
  --output-dir experiments/YOLO_Seg_Benchmark \
  --no-export-champion


#### 5.3 Modelo 3: `yolo26n-seg.pt` (SOTA Leve End-to-End NMS-Free — 3 Runs)


In [ ]:
!MPLBACKEND=Agg uv run python experiments/benchmark_yolo.py \
  --data datasets/generated/animals_video_v35_yolo_seg/data.yaml \
  --models yolo26n-seg.pt \
  --runs 3 --base-seed 42 --epochs 60 --imgsz 640 --batch 16 --patience 15 \
  --output-dir experiments/YOLO_Seg_Benchmark \
  --no-export-champion


#### 5.4 Modelo 4 *(Opcional — Escala Small)*: `yolo11s-seg.pt` (3 Runs)


In [ ]:
RUN_SMALL_OPTIONAL = False  # Mude para True se quiser incluir yolo11s-seg.pt na comparação
if RUN_SMALL_OPTIONAL:
    !MPLBACKEND=Agg uv run python experiments/benchmark_yolo.py \
      --data datasets/generated/animals_video_v35_yolo_seg/data.yaml \
      --models yolo11s-seg.pt \
      --runs 3 --base-seed 42 --epochs 60 --imgsz 640 --batch 16 --patience 15 \
      --output-dir experiments/YOLO_Seg_Benchmark \
      --no-export-champion
else:
    print("⏭️ Modelo opcional yolo11s-seg.pt ignorado (RUN_SMALL_OPTIONAL=False).")


---
### 6. Consolidação Estatística (`Média ± DP`, Variância, `N >= 3` vs `N <= 2`) & Exportação do Campeão (`640px` vs `320px`)


In [ ]:
import glob

executed_models = []
for ckpt, slug in [
    ("yolov8n-seg.pt", "yolov8n-seg"),
    ("yolo11n-seg.pt", "yolo11n-seg"),
    ("yolo26n-seg.pt", "yolo26n-seg"),
    ("yolo11s-seg.pt", "yolo11s-seg"),
]:
    if glob.glob(f"experiments/YOLO_Seg_Benchmark/{slug}/run*_seed*/test_metrics.json"):
        executed_models.append(ckpt)

models_arg = " ".join(executed_models)
print(f"📊 Consolidando modelos concluídos: {models_arg}")
!MPLBACKEND=Agg uv run python experiments/benchmark_yolo.py \
  --data datasets/generated/animals_video_v35_yolo_seg/data.yaml \
  --models {models_arg} \
  --output-dir experiments/YOLO_Seg_Benchmark \
  --consolidate-only


### 7. Visualização Inline do Relatório, Gráficos SOTA e Comparativo `640` vs `320`


In [ ]:
import json
from pathlib import Path
from IPython.display import display, Markdown, Image

bench_dir = Path("experiments/YOLO_Seg_Benchmark")
report_md = bench_dir / "yolo_benchmark_report.md"
if report_md.exists():
    display(Markdown(report_md.read_text(encoding="utf-8")))

champ_cmp = bench_dir / "champion" / "champion_640_vs_320.json"
if champ_cmp.exists():
    payload = json.loads(champ_cmp.read_text(encoding="utf-8"))
    print(f"🏆 Modelo Campeão: {payload.get('champion_model')}")
    for sz, info in payload.get("by_imgsz", {}).items():
        exps = {k: (f"{v['size_mb']}MB" if v.get("ok") else "falhou") for k, v in info.get("exports", {}).items()}
        print(
            f"  [imgsz={sz}] Mask mAP50-95={info['mask_map50_95']*100:.2f}% | "
            f"Box mAP50-95={info['box_map50_95']*100:.2f}% | "
            f"FP Fundo={info['bg_fp_image_rate']*100:.1f}% | "
            f"FPS={info['fps']:.1f} | Exports={exps}"
        )

for img_name in (
    "yolo_seg_benchmark_bars.png",
    "yolo_seg_pareto_fps.png",
    "yolo_seg_disjoint_vs_shared.png",
):
    p = bench_dir / img_name
    if p.exists():
        display(Image(filename=str(p)))


### 8. Amostras Visuais de Inferência no `split='test'` (Espécies Camufladas & Fundo Puro)


In [ ]:
import glob, json
from pathlib import Path
from ultralytics import YOLO
from IPython.display import display, Image

summary = json.loads(Path("experiments/YOLO_Seg_Benchmark/yolo_benchmark_summary.json").read_text(encoding="utf-8"))
champ_ckpt = summary["champion_model"]
best_pt = summary["models"][champ_ckpt]["best_run_pt"]
model_vis = YOLO(best_pt)

test_imgs = sorted(glob.glob("datasets/generated/animals_video_v35_yolo_seg/images/test/*.jpg"))
picks = []
for prefix in ("sapo_", "onca_", "arara_", "fundo_sem_animal_"):
    matches = [p for p in test_imgs if Path(p).name.startswith(prefix)]
    if matches:
        picks.append(matches[len(matches) // 2])

if picks:
    vis_dir = Path("experiments/YOLO_Seg_Benchmark/champion/visual_test_samples")
    model_vis.predict(source=picks, imgsz=640, conf=0.25, save=True, project=str(vis_dir), name="preds", exist_ok=True)
    for out_img in sorted((vis_dir / "preds").glob("*.jpg")):
        display(Image(filename=str(out_img), width=640))


### 9. Sincronização de Resultados (`DVC` + `GitHub`) & Backup no Drive


In [ ]:
!uv run dvc add experiments/YOLO_Seg_Benchmark || true
print("☁️ Enviando pesos e gráficos do benchmark YOLO-Seg para o Google Drive via DVC...")
!uv run dvc push -r gdrive-storage || true

!git config user.name "IFMaker Colab Runner"
!git config user.email "colab@ifmaker.edu.br"
!git add experiments/YOLO_Seg_Benchmark.dvc experiments/benchmark_yolo.py notebooks/Treinamento_YOLO_Colab_BioEcoInt.ipynb
!git commit -m "feat(yolo-seg): benchmark multi-run 3 seeds e exportacao campea 640 vs 320" || echo "Sem alterações para commit"

!git pull --rebase origin main || true
!git push origin main || echo "⚠️ Push rejeitado; artefatos salvos no DVC/Drive." 


### 10. Desconexão Automática Opcional da GPU


In [ ]:
from google.colab import runtime
AUTO_DISCONNECT = False  # Mude para True ao rodar sem supervisão
if AUTO_DISCONNECT:
    print("🔌 Encerrando runtime para economizar Compute Units...")
    runtime.unassign()
